<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Base de comercio bilateral y concentración de mercado

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `funciones.hhi`: cálculo del índice de concentración Herfindahl-Hirschman.

In [1]:
import pandas as pd
import numpy as np
from funciones.hhi import hhi

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

Define los territorios que se excluyen del análisis y el umbral mínimo de valor comercial.
Se excluyen las jurisdicciones que exportaron menos de 100 millones de dólares en 2023 y la
Federación de Rusia.

In [3]:
territorios_excluidos = [
    'AIA', 'ASM', 'ATA', 'ATF', 'BLM', 'BVT', 'CCK', 'COK', 'CXR', 'DMA', 'ESH',
    'GRD', 'GUM', 'HMD', 'IOT', 'KIR', 'LCA', 'MNP', 'MSR', 'NFK', 'NIU', 'PCN',
    'PLW', 'SGS', 'SHN', 'SPM', 'STP', 'SXM', 'TCA', 'TON', 'TUV', 'UMI', 'VAT',
    'VCT', 'WLF', 'WSM'
]

valor_minimo = 50   # miles de dólares

### Clasificación geoeconómica del importador

Asigna cada código ISO3 de país importador a uno de siete grupos: Unión Europea, Reino Unido,
APEC Asia, Estados Unidos, Canadá, América Latina y el Caribe, y Otro.

In [4]:
def clasificar_pais(p):
    if p in {'AUT', 'BEL', 'BGR', 'HRV', 'CYP', 'CZE', 'DNK', 'EST', 'FIN', 'FRA', 'DEU',
      'GRC', 'HUN', 'IRL', 'ITA', 'LVA', 'LTU', 'LUX', 'MLT', 'NLD', 'POL', 'PRT',
      'ROM', 'SVK', 'SVN', 'ESP', 'SWE'}:
        return 'Unión Europea'
    elif p in {'GBR'}:
        return 'Reino Unido'
    elif p in {'CHN', 'HKG', 'IDN', 'JPN', 'KOR', 'MYS', 'PHL', 'SGP', 'TWN', 'THA', 'VNM', 'RUS'}:
        return 'APEC Asia'
    elif p in {'USA'}:
        return 'Estados Unidos'
    elif p in {'CAN'}:
        return 'Canadá'
    elif p in {'ARG', 'BHS', 'BLZ', 'BOL', 'BRA', 'CHL', 'COL', 'CRI', 'CUB', 'DMA', 'DOM', 'ECU',
       'GRD', 'GTM', 'GUY', 'HND', 'JAM', 'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV',
       'TTO', 'URY', 'VCT'}:
        return 'América Latina y el Caribe'
    else:
        return 'Otro'

### Base de comercio con jurisdicciones de interés

Prepara la base de comercio bilateral de WITS.

**Pasos realizados:**

1. **Filtrado:** conserva las observaciones con valor de al menos 50 mil dólares y cantidad positiva, y excluye a Rusia y a los territorios listados en los parámetros.

2. **Estandarización:** completa el código HS6 a seis dígitos y renombra las columnas al español.

3. **Clasificación del importador:** asigna cada país importador a su grupo geoeconómico.

4. **Variable adicional:** extrae el código HS4 de los primeros cuatro dígitos del HS6.

In [5]:
df_comercio = pd.read_parquet(raw / 'comercio' / 'wits.parquet').query("`TradeValue in 1000 USD` >= @valor_minimo and Quantity > 0 and PartnerISO3 != 'RUS' and ReporterISO3 != 'RUS' and (PartnerISO3.isin(@territorios_excluidos)) == False and (ReporterISO3.isin(@territorios_excluidos)) == False")
df_comercio['ProductCode'] = df_comercio['ProductCode'].astype(str).str.zfill(6)
df_comercio = df_comercio[['Year', 'PartnerISO3', 'PartnerName', 'ReporterISO3', 'ReporterName', 'ProductCode', 'TradeValue in 1000 USD', 'Quantity']].rename(columns={'Year':'Año','ReporterISO3':'ISO Importador','PartnerISO3':'ISO Exportador','ReporterName':'Importador','PartnerName':'Exportador','Quantity':'Cantidad','ProductCode':'HS6','TradeValue in 1000 USD':'Valor Comercio'})

df_comercio['Grupo Importador'] = df_comercio['ISO Importador'].apply(clasificar_pais)
df_comercio['HS4'] = df_comercio['HS6'].str[:4]
df_comercio = df_comercio[['Año', 'ISO Exportador', 'Exportador', 'ISO Importador', 'Importador','Grupo Importador','HS4','HS6', 'Valor Comercio', 'Cantidad']].reset_index(drop=True)

temp.mkdir(parents=True, exist_ok=True)
df_comercio.to_parquet(temp / 'df_comercio.parquet', index=False)

### Base de comercio mundial

Estandariza el código de producto a seis dígitos y aplica la misma exclusión de territorios.
Esta base alimenta la estimación del espacio de producto.

In [6]:
df_comercio_mundial = pd.read_parquet(raw / 'comercio' / 'df_comercio_mundial.parquet')
df_comercio_mundial['ProductCode'] = df_comercio_mundial['ProductCode'].astype(str).str.zfill(6)
df_comercio_mundial = df_comercio_mundial.query("(PartnerISO3.isin(@territorios_excluidos)) == False and (ReporterISO3.isin(@territorios_excluidos)) == False")

temp.mkdir(parents=True, exist_ok=True)
df_comercio_mundial.to_parquet(temp / 'df_comercio_mundial.parquet', index=False)

### Concentración de mercado

Estima el índice de Herfindahl-Hirschman por año, producto y mercado de destino, tomando como
entidad al país exportador. Excluye las exportaciones de Colombia para medir la estructura
competitiva internacional que enfrenta el país.

In [7]:
df_hhi = hhi(df_comercio.query("`ISO Exportador` != 'COL'"),
    grupo_entidad="ISO Exportador",
    valor="Valor Comercio",
    otras_dimensiones=["Año", "HS6", "ISO Importador"],
    renombrar_columna="HHI")

temp.mkdir(parents=True, exist_ok=True)
df_hhi.to_parquet(temp / 'df_hhi.parquet', index=False)